In [1]:
import os 
from pathlib import Path 
import pandas as pd 
import numpy as np 

from xgboost import XGBClassifier

In [2]:
os.chdir("..")

In [3]:
print(os.getcwd())

e:\project_archive\new project


In [7]:
# Load data

data_path = Path("data/processed")
if not data_path.exists():
    raise FileNotFoundError

X_train = pd.read_csv(data_path / "x_train.csv")
y_train = np.load(data_path / "y_train.npy")
print("X_train and y_train loaded successfully")

X_test = pd.read_csv(data_path / "x_test.csv")
y_test = np.load(data_path / "y_test.npy")
print("X_test and y_test loaded successfully")

X_train and y_train loaded successfully
X_test and y_test loaded successfully


In [5]:
X_train.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP
0,1,17,5,9119,1,1,130.0,1,19,1,...,0,0,5,14,1,11.000000,0,8.9,1.4,3.51
1,1,1,6,9500,1,1,133.0,1,1,1,...,0,1,8,11,8,12.361111,0,12.4,0.5,1.79
2,1,17,3,9147,1,1,118.0,1,1,19,...,0,0,5,7,5,13.200000,0,12.4,0.5,1.79
3,1,1,1,171,1,1,124.0,1,1,19,...,0,0,0,0,0,0.000000,0,13.9,-0.3,0.79
4,1,16,5,9085,1,1,134.0,1,19,37,...,2,0,5,7,5,12.400000,0,9.4,-0.8,-3.12


In [6]:
y_train

array([0, 2, 1, ..., 2, 2, 0], shape=(3096,))

In [9]:
import yaml 

config_path = Path("config")
with open(config_path / "model_params.yaml", "r") as f:
    config = yaml.safe_load(f)

print(f"config file loaded")
print(config["XGBoost"])

config file loaded
{'n_estimators': {'min': 100, 'max': 500}, 'learning_rate': {'min': 0.01, 'max': 0.3, 'log': True}, 'max_depth': {'min': 3, 'max': 10}, 'min_child_weight': {'min': 1, 'max': 10}, 'gamma': {'min': 0.0, 'max': 5.0}, 'subsample': {'min': 0.6, 'max': 1.0}, 'colsample_bytree': {'min': 0.6, 'max': 1.0}, 'reg_alpha': {'min': 0.0, 'max': 10.0}, 'reg_lambda': {'min': 0.1, 'max': 20.0, 'log': True}, 'objective': ['multi:softprob'], 'eval_metric': ['mlogloss'], 'tree_method': ['hist'], 'grow_policy': ['depthwise'], 'booster': ['gbtree']}


In [10]:
def suggest_params(trial, config):
    return  {
    "n_estimators": trial.suggest_int(
        "n_estimators",
        config["XGBoost"]["n_estimators"]["min"],
        config["XGBoost"]["n_estimators"]["max"],
        step=50,
    ),

    "learning_rate": trial.suggest_float(
        "learning_rate",
        config["XGBoost"]["learning_rate"]["min"],
        config["XGBoost"]["learning_rate"]["max"],
        log=config["XGBoost"]["learning_rate"]["log"],
    ),

    "max_depth": trial.suggest_int(
        "max_depth",
        config["XGBoost"]["max_depth"]["min"],
        config["XGBoost"]["max_depth"]["max"],
    ),

    "min_child_weight": trial.suggest_int(
        "min_child_weight",
        config["XGBoost"]["min_child_weight"]["min"],
        config["XGBoost"]["min_child_weight"]["max"],
    ),

    "gamma": trial.suggest_float(
        "gamma",
        config["XGBoost"]["gamma"]["min"],
        config["XGBoost"]["gamma"]["max"],
    ),

    "subsample": trial.suggest_float(
        "subsample",
        config["XGBoost"]["subsample"]["min"],
        config["XGBoost"]["subsample"]["max"],
    ),

    "colsample_bytree": trial.suggest_float(
        "colsample_bytree",
        config["XGBoost"]["colsample_bytree"]["min"],
        config["XGBoost"]["colsample_bytree"]["max"],
    ),

    "reg_alpha": trial.suggest_float(
        "reg_alpha",
        config["XGBoost"]["reg_alpha"]["min"],
        config["XGBoost"]["reg_alpha"]["max"],
    ),

    "reg_lambda": trial.suggest_float(
        "reg_lambda",
        config["XGBoost"]["reg_lambda"]["min"],
        config["XGBoost"]["reg_lambda"]["max"],
        log=config["XGBoost"]["reg_lambda"]["log"],
    ),

    "objective": config["XGBoost"]["objective"][0],
    "eval_metric": config["XGBoost"]["eval_metric"][0],
    "tree_method": config["XGBoost"]["tree_method"][0],
    "grow_policy": config["XGBoost"]["grow_policy"][0],
    "booster": config["XGBoost"]["booster"][0],

    "random_state": 42,
    "n_jobs": -1,
}

In [ ]:
import optuna
import wandb

from sklearn.model_selection import StratifiedKFold, cross_val_score

def objective(trial, X, y, config, run):

    params = suggest_params(
        trial=trial,
        config=config
    )

    model = XGBClassifier(
        **params
        )

    cv_strategy = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    scores = cross_val_score(
        estimator=model,
        X=X,
        y=y,
        cv=cv_strategy,
        scoring="f1_macro",
        n_jobs=-1,
    )

    score = scores.mean()

    run.log(
        {
            "trial": trial.number,
            "f1_macro_avg": score,
        }
    )

    return score

e:\project_archive\new project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# train.py
from sklearn.metrics import f1_score

with wandb.init(
        project="student-drop-enroll-grad-preds",
        name="XGboost-Optuna-v1",
        group="XGBoost",
        tags=["XGBoost", "Optuna"]
    ) as run:
     
      study = optuna.create_study(
            direction='maximize'
      )

      study.optimize(
            lambda trial: objective(
                  trial,
                  X_train,
                  y_train,
                  config=config,
                  run=run
            ),
            n_trials=80
      )

      best_params = study.best_params

      model = XGBClassifier(
            **best_params
      )

      model.fit(X_train, y_train)

      pred = model.predict(X_test)

      f1 = f1_score(
            y_test,
            pred,
            average='macro'
      )

      run.summary["best_cv_score"] = study.best_value
      run.summary["test_f1_macro"] = f1
      run.summary["best_params"] = best_params



wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Asus\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


[I 2026-06-26 13:43:21,858] A new study created in memory with name: no-name-4f6bd2c6-0b04-4273-9043-0145ce7ac7a5
[I 2026-06-26 13:43:28,450] Trial 0 finished with value: 0.7007423796530878 and parameters: {'n_estimators': 250, 'learning_rate': 0.05814918079580541, 'max_depth': 9, 'min_child_weight': 5, 'gamma': 1.207159844669297, 'subsample': 0.9232779232566115, 'colsample_bytree': 0.8282184421311105, 'reg_alpha': 1.3049698596082182, 'reg_lambda': 0.3148831841609996}. Best is trial 0 with value: 0.7007423796530878.
[I 2026-06-26 13:43:32,282] Trial 1 finished with value: 0.6602420618796121 and parameters: {'n_estimators': 200, 'learning_rate': 0.05061979165230433, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 2.895606711136188, 'subsample': 0.8741242009648963, 'colsample_bytree': 0.7893079214667987, 'reg_alpha': 6.211007422272354, 'reg_lambda': 0.5404734153681734}. Best is trial 0 with value: 0.7007423796530878.
[I 2026-06-26 13:43:33,162] Trial 2 finished with value: 0.665780512774

f1_macro_avg,▇▃▄▄▁▇▇▅▆▇▇▇▆█▇▇▆▇▆█▇▆█▅▄▆▇▃▆▅██▇█▇█████
trial,▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇████
best_cv_score,0.71256
f1_macro_avg,0.70872
test_f1_macro,0.68786
trial,79


In [13]:
study.best_params

{'n_estimators': 250,
 'learning_rate': 0.04975825394678496,
 'max_depth': 9,
 'min_child_weight': 6,
 'gamma': 0.6728775289607527,
 'subsample': 0.6296668591084468,
 'colsample_bytree': 0.8363491995081975,
 'reg_alpha': 0.8809018434573517,
 'reg_lambda': 0.35375688072013056}

In [14]:
study.best_value

0.7125592715562751

In [15]:
save_result_path = Path("artifacts/best_params")

save_result_path.mkdir(parents=True, exist_ok=True)

result = {
    "model_name": "XGBoost",
    "best_trial": study.best_trial.number,
    "best_params": study.best_params,
    "best_value": float(study.best_value)
}

with open(save_result_path / "xgboost.yaml", "w") as f:
    yaml.safe_dump(result, f, sort_keys=False)
